In [ ]:
from faster_whisper import WhisperModel
import json

# 1. Cấu hình model (dùng bản nhỏ 'tiny' hoặc 'base' để test nhanh, khi thi đổi sang 'large-v3')
model_size = "base"
# Nếu chạy bằng GPU thì chọn device="cuda", nếu chạy CPU thì chọn device="cpu"
model = WhisperModel(model_size, device="cpu", compute_type="int8")

video_audio_path = r"sample-speech-1m.mp3" # Đường dẫn file audio trích từ clip test

print("--- Đang xử lý Audio bằng Whisper ---")
segments, info = model.transcribe(video_audio_path, beam_size=5, language="vi")

# 2. Thu thập kết quả text kèm timestamp
audio_text_data = []
for segment in segments:
    print(f"[{segment.start:.2f}s -> {segment.end:.2f}s]: {segment.text}")
    audio_text_data.append({
        "start": segment.start,
        "end": segment.end,
        "text": segment.text.strip()
    })

# 3. Giả định dữ liệu Scene nhận được từ Phúc để test thuật toán Chunking
# Ví dụ Phúc báo về video có 2 scene với mốc thời gian như sau:
scenes_from_phuc = [
    {"scene_id": 1, "start_time": 0.0, "end_time": 60.16}
]

# 4. Thuật toán Chunking của Khoa: Khớp Text vào đúng Scene
processed_scenes = []
for scene in scenes_from_phuc:
    scene_text = []
    for text_block in audio_text_data:
        # Kiểm tra nếu đoạn text nằm trong khoảng thời gian của scene (cho phép lệch nhẹ nếu cần)
        if text_block["start"] >= scene["start_time"] and text_block["end"] <= scene["end_time"]:
            scene_text.append(text_block["text"])
    
    # Gộp các câu thoại trong cùng một scene lại với nhau
    full_scene_script = " ".join(scene_text)
    
    processed_scenes.append({
        "scene_id": scene["scene_id"],
        "time_range": [scene["start_time"], scene["end_time"]],
        "text_from_audio": full_scene_script
    })

# 5. Xuất thử cấu trúc sơ khai để chuyển cho Khôi thiết kế JSON chính thức
print("\n--- Kết quả sau khi Chunking theo Scene ---")
print(json.dumps(processed_scenes, ensure_ascii=False, indent=4))

--- Đang xử lý Audio bằng Whisper ---
[0.00s -> 7.24s]:  Welcome to samplolid.com, a free online resource for downloading sample files in a wide variety of digital formats.
[7.24s -> 14.84s]:  Whether you are a software developer testing file upload functionality, a quality assurance engineer validating media players,
[14.84s -> 20.16s]:  a student learning about digital formats, or simply someone who needs a quick test file,
[20.16s -> 25.92s]:  samplolid provides ready to use files that you can download instantly, completely free of charge.
[25.92s -> 30.44s]:  In this recording, we will walk you through every aspect of the samplolid platform.
[30.44s -> 34.92s]:  Exploring the formats we offer, the technical details behind each one,
[34.92s -> 39.12s]:  and the many ways these test files can be used in your projects and workflows.
[39.12s -> 41.20s]:  Let us begin with image formats.
[41.20s -> 44.88s]:  Images are perhaps the most fundamental type of digital media,
[44.88s -> 52.76